# artificial-sort — Poset RL Training Environment

Learn to sort with the fewest pairwise comparisons using reinforcement learning.

**Backends:** PyTorch (CPU/GPU/MPS) · JAX/Flax (CPU/GPU/TPU)  
**Colab GPU:** *Runtime → Change runtime type → T4 GPU*  
**Colab TPU:** *Runtime → Change runtime type → TPU v2*

## 1 · Install

In [ ]:
import subprocess, sys

def _pip(*a):
    subprocess.check_call([sys.executable, "-m", "pip", "install", "-q", *a])

# install package + all optional deps (torch + jax + plotting)
_pip("artificial-sort[notebook] @ git+https://github.com/blackgauss/artificial-sort.git@feat/attention-policy")
# _pip("-e", "..")  # ← uncomment when working from a local clone

import math, time
import numpy as np
import matplotlib.pyplot as plt

print("Install complete")

## 2 · Detect Hardware

In [ ]:
import torch, jax

# ── PyTorch device ────────────────────────────────────────────────────────
if torch.cuda.is_available():
    TORCH_DEVICE = "cuda"
    print(f"PyTorch  ✓  CUDA  {torch.cuda.get_device_name(0)}")
elif hasattr(torch.backends, "mps") and torch.backends.mps.is_available():
    TORCH_DEVICE = "mps"
    print("PyTorch  ✓  Apple MPS")
else:
    TORCH_DEVICE = "cpu"
    print("PyTorch  ·  CPU")

# ── JAX device ────────────────────────────────────────────────────────────
jax_devs = jax.devices()
JAX_DEVICE = jax_devs[0].platform   # 'cpu' | 'gpu' | 'tpu'
print(f"JAX      ✓  {JAX_DEVICE.upper()}  × {len(jax_devs)}  ({jax.__version__})")

## 3 · Choose an Experiment Config

Edit the dataclass inline **or** load a YAML from `configs/`.  
To add a new architecture: drop a file in `poset_rl/models/` with `@register("name")` — nothing else changes.

| model name | framework | notes |
|---|---|---|
| `"mlp"` | PyTorch | fixed n |
| `"attention"` | PyTorch | any n, curriculum |
| `"jax_mlp"` | JAX/Flax | fixed n |
| `"jax_attention"` | JAX/Flax | any n, curriculum, TPU-ready |

In [ ]:
from poset_rl import ExperimentConfig, ModelConfig, TrainConfig, EvalConfig, list_models

print("Registered models:", list_models())

# ── pick your config here ─────────────────────────────────────────────────
cfg = ExperimentConfig(
    name  = "jax_attention_curriculum",
    model = ModelConfig(name="jax_attention", hidden=64, nhead=2, nlayers=1),
    train = TrainConfig(
        n_or_range   = [3, 4, 5, 6],
        episodes     = 3000,
        lr           = 3e-3,
        log_interval = 500,
        out_csv      = "training.csv",
        seed         = 42,
    ),
    eval  = EvalConfig(ns=[3, 4, 5, 6], episodes=300),
)

# ── or load from a yaml ───────────────────────────────────────────────────
# cfg = ExperimentConfig.from_yaml("configs/jax_attention_curriculum.yaml")

print(f"\nExperiment : {cfg.name}")
print(f"Model      : {cfg.model.name}  hidden={cfg.model.hidden}")
print(f"Framework  : {cfg.model.framework}")
print(f"n_or_range : {cfg.train.n_or_range}  episodes={cfg.train.episodes}")

## 4 · Train

`train_from_config` dispatches automatically:  
- `jax_*` models → JAX loop with `optax.adam` + `nnx.value_and_grad`  
- all other models → PyTorch loop with `torch.optim.Adam`

In [ ]:
from poset_rl.train import train_from_config

t0 = time.time()
model, history = train_from_config(cfg)
print(f"\nDone in {time.time()-t0:.1f}s  ({len(history)} episodes)")

## 5 · Benchmark

In [ ]:
from poset_rl.bench import agent_random, agent_greedy, evaluate_baseline, evaluate_model

ns  = cfg.eval.ns
eps = cfg.eval.episodes

rand_res   = evaluate_baseline(agent_random, ns, eps)
greedy_res = evaluate_baseline(agent_greedy, ns, eps)

# JAX models use .act(); PyTorch models use .select_action() — bench handles both
model_res  = evaluate_model(model, ns, eps, device=TORCH_DEVICE if cfg.model.framework == "torch" else None)

print(f"\n{'n':>3}  {'lb':>4}  {'random':>8}  {'greedy':>8}  {cfg.model.name:>14}")
print("-" * 50)
for n in ns:
    lb = math.ceil(math.log2(math.factorial(n)))
    print(f"{n:>3}  {lb:>4}  {rand_res[n]:8.2f}  {greedy_res[n]:8.2f}  {model_res[n]:14.2f}")
print("\nlb = ⌈log₂(n!)⌉ (information-theoretic minimum)")

## 6 · Plot

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 5))
W = 100

# ── learning curve ─────────────────────────────────────────────────────────
ax = axes[0]
steps = [r["steps"] for r in history]
ax.plot(np.convolve(steps, np.ones(W)/W, "valid"), color="tomato", label=cfg.name)
for n in cfg.train.n_or_range if isinstance(cfg.train.n_or_range, list) else [cfg.train.n_or_range]:
    lb = math.ceil(math.log2(math.factorial(n)))
    ax.axhline(lb, linestyle="--", linewidth=0.7, alpha=0.5, label=f"lb n={n}")
ax.set_xlabel("Episode"); ax.set_ylabel("Comparisons")
ax.set_title(f"Learning curve — {cfg.name}"); ax.legend(fontsize=8); ax.grid(True, alpha=0.3)

# ── per-n bar chart ────────────────────────────────────────────────────────
ax = axes[1]
x, w = np.arange(len(ns)), 0.2
lbs  = [math.ceil(math.log2(math.factorial(n))) for n in ns]
ax.bar(x - 1.5*w, lbs,                            w, label="lower bound",    color="gold",          alpha=0.8)
ax.bar(x - 0.5*w, [rand_res[n]   for n in ns],    w, label="random",         color="lightgray",     alpha=0.9)
ax.bar(x + 0.5*w, [greedy_res[n] for n in ns],    w, label="greedy",         color="mediumseagreen",alpha=0.9)
ax.bar(x + 1.5*w, [model_res[n]  for n in ns],    w, label=cfg.model.name,   color="tomato",        alpha=0.9)
ax.set_xticks(x); ax.set_xticklabels([f"n={n}" for n in ns])
ax.set_ylabel("Mean comparisons"); ax.set_title("Agent comparison")
ax.legend(fontsize=8); ax.grid(True, axis="y", alpha=0.3)

plt.tight_layout()
plt.savefig("benchmark.png", dpi=150)
plt.show()